In [ ]:
import openai
import re

client = openai.OpenAI(
    api_key="s.....................",
    base_url="https://api.futurexailab.com"
)

bias_keywords = {
    "gender": ["women should", "men should", "girls can't", "boys can't", "because they are women", "because they are men"],
    "race": ["black people", "white people are", "asians are", "race is"],
    "age": ["old people can't", "young people can't", "too old to", "too young to"],
    "role_stereotype": ["nurses should be women", "leaders should be men", "tech jobs are for men"]
}

def detect_bias(text):
    detected = []
    text_low = text.lower()

    for category, patterns in bias_keywords.items():
        for p in patterns:
            if p.lower() in text_low:
                detected.append((category, p))
    return detected


def explain_bias(text, bias_list):
    if not bias_list:
        return "No clear bias detected."

    prompt = f"""
Sentence: "{text}"
Detected Bias: {bias_list}

Explain clearly why these phrases are biased and provide a neutral alternative.
"""

    try:
        response = client.chat.completions.create(
            model="gemini-2.5-flash",
            messages=[
                {"role": "system", "content": "You are a helpful bias-analysis assistant."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=300
        )

        return response.choices[0].message.content

    except openai.RateLimitError:
        return "Error: Too many requests or quota exceeded."
    except openai.OpenAIError as e:
        return f"OpenAI API error: {e}"
    except Exception as e:
        return f"Unexpected error: {e}"


# ---------------------------
# MAIN LOOP
# ---------------------------
print("AI Bias Detector (type 'exit' to quit)")

while True:
    user_input = input("\nEnter a sentence: ")

    if user_input.lower() == "exit":
        print("Goodbye!")
        break

    bias_found = detect_bias(user_input)
    result = explain_bias(user_input, bias_found)

    print("\n--- Result ---")
    print(result)


AI Bias Detector (type 'exit' to quit)

Enter a sentence: Nurses should be women.

--- Result ---
None

Enter a sentence: young people can't

--- Result ---
None

Enter a sentence: exit
Goodbye!
